# 13. VLM 분석에서 나온 개선 가설 검증(H1~H3)

11번 노트북(VLM 오류 분석)에서 세 가설을 얻었습니다. 판정 기준은 실행 전에 `experiments/IMP/PLAN.md`에 적었습니다.

| 가설 | VLM이 든 요인 | 확인 방법 |
|---|---|---|
| H1 | 한 순간만 담김(40장 중 36) | 같은 클립 프레임의 확률 평균(clip8, cam4) |
| H2 | 사람이 작음(33) | ccrop의 CV1 판정(학습 결과) |
| H3 | 기구 식별(30) | 흑백 입력 진단, 흑백 학습 비교, Grad-CAM |

- **새로 학습하지 않습니다.** 저장한 가중치로 추론만 합니다(`src/imp_predict.py`). Test는 쓰지 않습니다.
- CV fold 점수는 그 fold 검증 사람(클래스마다 2명 이상)에 대한 것입니다. Early Stopping도 같은 데이터로 했으므로 절대 성능이 아니라 비교용입니다.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, recall_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from common import FIGURES, class_names, set_korean_font

print("font:", set_korean_font())
pd.set_option("display.width", 220)
NAMES = class_names()
LABELS = list(range(12))
P = ROOT / "experiments" / "IMP" / "pred"
PC = [f"p{c}" for c in LABELS]
def load(run, pre="train"):
    return pd.read_csv(P / f"{run}__{pre}.csv")
def smooth(d, keys):
    return d.groupby(keys)[PC].transform("mean").values.argmax(1)
def mf1(y, p):
    return f1_score(y, p, labels=LABELS, average="macro", zero_division=0)
def rec(y, p):
    return pd.Series(recall_score(y, p, labels=LABELS, average=None, zero_division=0), index=LABELS)
print(sorted(x.name for x in P.glob("*.csv")))

font: Malgun Gothic
['CV1-ccrop-f1__train.csv', 'CV1-ccrop-f2__train.csv', 'CV1-ccrop-f3__train.csv', 'CV1-gray-f1__train.csv', 'CV1-gray-f2__train.csv', 'CV1-gray-f3__train.csv', 'CV1-letterbox-f1__gray.csv', 'CV1-letterbox-f1__train.csv', 'CV1-letterbox-f2__gray.csv', 'CV1-letterbox-f2__train.csv', 'CV1-letterbox-f3__gray.csv', 'CV1-letterbox-f3__train.csv', 'E04__train.csv', 'SEL-A-s43__train.csv', 'SEL-A-s44__train.csv']


## H1. 클립 확률 평균

- **이미지**: 사진 한 장의 예측(기본 과제)
- **clip8**: 같은 클립 8장(시점 4개 × 시점마다 다른 카메라 2대, 카메라는 5대)의 확률 평균으로 바꾼 예측
- **cam4**: 같은 클립이면서 같은 카메라인 사진의 평균. 계획 때는 4장으로 알았지만 실제로는 **카메라마다 1~2장**입니다(15번 노트북 A3에서 확인). 이름은 기록과 맞추려고 그대로 둡니다
- 점수는 모두 같은 이미지 집합의 Macro F1입니다. 추론 방식을 바꾸는 것이라 기본 결과와 섞지 않고 따로 봅니다.

In [2]:
rows = []
for run in [f"CV1-letterbox-f{f}" for f in (1, 2, 3)] + [f"CV1-ccrop-f{f}" for f in (1, 2, 3)] + ["E04", "SEL-A-s43", "SEL-A-s44"]:
    d = load(run)
    p8, p4 = smooth(d, "clip_id"), smooth(d, ["clip_id", "camera"])
    agree = d.groupby("clip_id").pred.nunique().eq(1).mean()
    r0, r8 = rec(d.target_id, d.pred), rec(d.target_id, p8)
    rows.append({"모델": run, "검증": "공식 Val" if not run.startswith("CV1") else "fold " + run[-1],
                 "이미지 F1": mf1(d.target_id, d.pred), "clip8 F1": mf1(d.target_id, p8), "cam4 F1": mf1(d.target_id, p4),
                 "8장 예측이 모두 같은 클립": agree,
                 "덤벨 로우 R": f"{r0[2]:.2f}→{r8[2]:.2f}", "데드리프트 R": f"{r0[7]:.2f}→{r8[7]:.2f}", "바벨 로우 R": f"{r0[9]:.2f}→{r8[9]:.2f}"})
h1 = pd.DataFrame(rows).set_index("모델")
h1["Δ clip8"] = h1["clip8 F1"] - h1["이미지 F1"]
h1["Δ cam4"] = h1["cam4 F1"] - h1["이미지 F1"]
h1.round(4)

,검증,이미지 F1,clip8 F1,cam4 F1,8장 예측이 모두 같은 클립,덤벨 로우 R,데드리프트 R,바벨 로우 R,Δ clip8,Δ cam4
모델,,,,,,,,,,
CV1-letterbox-f1,fold 1,0.8193,0.8789,0.8300,0.6969,0.45→0.68,0.88→1.00,0.37→0.40,0.0596,0.0107
CV1-letterbox-f2,fold 2,0.8730,0.9190,0.8869,0.6964,0.85→1.00,0.60→0.57,0.30→0.50,0.0460,0.0139
CV1-letterbox-f3,fold 3,0.8423,0.8789,0.8421,0.7167,0.79→1.00,0.44→0.68,0.27→0.12,0.0366,-0.0002
CV1-ccrop-f1,fold 1,0.9009,0.9240,0.9079,0.7947,0.82→1.00,0.89→1.00,0.26→0.13,0.0231,0.0070
CV1-ccrop-f2,fold 2,0.8759,0.8992,0.8849,0.7577,0.87→0.95,0.77→0.80,0.16→0.12,0.0233,0.0091
CV1-ccrop-f3,fold 3,0.8649,0.8901,0.8753,0.8095,0.88→1.00,0.87→1.00,0.02→0.00,0.0253,0.0105
E04,공식 Val,0.8581,0.9082,0.8782,0.6389,0.79→1.00,0.40→0.35,0.53→0.88,0.0501,0.0200
SEL-A-s43,공식 Val,0.8782,0.9487,0.8999,0.6278,0.78→1.00,0.59→0.70,0.48→0.75,0.0705,0.0218
SEL-A-s44,공식 Val,0.8845,0.9404,0.9069,0.6556,0.91→1.00,0.54→0.55,0.53→0.88,0.0559,0.0224


In [3]:
main = h1.loc[[f"CV1-letterbox-f{f}" for f in (1, 2, 3)], "Δ clip8"]
pos, neg = int((main > 0).sum()), int((main < 0).sum())
verdict = "지지" if (main.mean() >= 0.010 and pos >= 2) else ("반대" if neg >= 2 else "차이 없음")
print(f"H1 주 판정(CV1-letterbox, clip8): fold별 Δ {main.round(4).tolist()}, 평균 {main.mean():+.4f}, 개선 fold {pos}개 → {verdict}")

H1 주 판정(CV1-letterbox, clip8): fold별 Δ [0.0596, 0.046, 0.0366], 평균 +0.0474, 개선 fold 3개 → 지지


## H2. 사람을 크게(ccrop): CV1 1단계 판정

새 기준을 만들지 않고 `experiments/CV/PLAN.md`의 1단계 분류를 그대로 씁니다(12번 노트북과 같은 계산).

In [4]:
EXP = ROOT / "experiments"
def cv_f1(name):
    return pd.Series({f: json.loads((EXP / f"CV1-{name}-f{f}" / "full" / "summary.json").read_text(encoding="utf-8"))["val"]["macro_f1"] for f in (1, 2, 3)})
def cv_t(name):
    return np.mean([json.loads((EXP / f"CV1-{name}-f{f}" / "full" / "summary.json").read_text(encoding="utf-8"))["mean_train_epoch_s"] for f in (1, 2, 3)])
b, c = cv_f1("letterbox"), cv_f1("ccrop")
dl = c - b
pos, neg, m = int((dl > 0).sum()), int((dl < 0).sum()), dl.mean()
t_ratio = cv_t("ccrop") / cv_t("letterbox")
if pos == 3: cls = "일관 개선형"
elif pos >= 2 and m >= 0.010: cls = "평균 개선형"
elif m > -0.010 and t_ratio <= 0.7: cls = "효율 개선형"
elif neg >= 2 and m <= -0.010: cls = "나쁨"
else: cls = "차이 없음" + (" (재검증 예외)" if (m >= 0.005 and pos >= 2) else "")
print("letterbox:", b.round(4).tolist(), " ccrop:", c.round(4).tolist())
print(f"Δ_f {dl.round(4).tolist()}, 평균 {m:+.4f}, 시간 비율 {t_ratio:.2f} → CV1 분류: {cls}")
print("P1(공식 Val, seed 3개) 판정은 '좋음'(+0.022)이었습니다.")

letterbox: [0.8193, 0.873, 0.8423]  ccrop: [0.9009, 0.8759, 0.8649]
Δ_f [0.0817, 0.0028, 0.0225], 평균 +0.0357, 시간 비율 1.02 → CV1 분류: 일관 개선형
P1(공식 Val, seed 3개) 판정은 '좋음'(+0.022)이었습니다.


## H3. 색 단서

- **방법 1**: 컬러로 학습한 `CV1-letterbox` 모델에 같은 사진을 흑백(3채널 복제)으로 넣습니다. 처음 보는 입력이라 모든 클래스가 떨어질 수 있으므로 **클래스 사이의 하락 순위**를 봅니다.
- **방법 2**: 처음부터 흑백으로 학습한 `CV1-gray`와 `CV1-letterbox`의 클래스별 Recall 차이입니다. 모델이 흑백에 적응한 뒤에도 남는 손실입니다.

In [5]:
def drops(pre_a, run_a, pre_b, run_b):
    out = []
    for f in (1, 2, 3):
        a, bb = load(f"{run_a}-f{f}", pre_a), load(f"{run_b}-f{f}", pre_b)
        out.append(rec(a.target_id, a.pred) - rec(bb.target_id, bb.pred))
    return pd.concat(out, axis=1).mean(axis=1)
m1 = drops("train", "CV1-letterbox", "gray", "CV1-letterbox")   # 컬러 − 흑백 입력
m2 = drops("train", "CV1-letterbox", "train", "CV1-gray")       # 컬러 학습 − 흑백 학습
h3 = pd.DataFrame({"방법1 하락(컬러−흑백 입력)": m1, "방법1 순위": m1.rank(ascending=False, method="min").astype(int),
                   "방법2 하락(컬러 학습−흑백 학습)": m2, "방법2 순위": m2.rank(ascending=False, method="min").astype(int)})
h3.index = [f"{c} {NAMES[c]}" for c in LABELS]
for pre, run in [("train", "CV1-letterbox"), ("gray", "CV1-letterbox"), ("train", "CV1-gray")]:
    print(run, pre, "Macro F1 fold 평균:", round(np.mean([mf1(load(f"{run}-f{f}", pre).target_id, load(f"{run}-f{f}", pre).pred) for f in (1, 2, 3)]), 4))
h3.round(3)

CV1-letterbox train Macro F1 fold 평균: 0.8449
CV1-letterbox gray Macro F1 fold 평균: 0.6452
CV1-gray train Macro F1 fold 평균: 0.8457


,방법1 하락(컬러−흑백 입력),방법1 순위,방법2 하락(컬러 학습−흑백 학습),방법2 순위
0 Y - Exercise,0.073,7,-0.026,11
1 니푸쉬업,-0.001,10,-0.001,6
2 덤벨 벤트오버 로우,0.160,5,-0.016,9
3 딥스,0.726,1,-0.001,5
4 라잉 레그 레이즈,0.538,2,-0.046,12
5 랫풀 다운,0.100,6,-0.004,7
6 로잉머신,0.000,8,0.000,4
7 바벨 데드리프트,0.172,4,-0.024,10
8 바벨 런지,0.239,3,0.021,3
9 바벨 로우,-0.055,12,0.066,1


In [6]:
d2 = h3.iloc[2]
ok = lambda drop, rank: drop >= 0.10 and rank <= 3  # noqa: E731
weak = lambda drop: drop < 0.05  # noqa: E731
s1, s2 = ok(d2.iloc[0], d2.iloc[1]), ok(d2.iloc[2], d2.iloc[3])
v = "지지" if (s1 and s2) else ("반대" if (weak(d2.iloc[0]) and weak(d2.iloc[2])) else "부분 지지")
print(f"H3(덤벨 로우): 방법1 하락 {d2.iloc[0]:+.3f}(순위 {int(d2.iloc[1])}), 방법2 하락 {d2.iloc[2]:+.3f}(순위 {int(d2.iloc[3])}) → {v}")

H3(덤벨 로우): 방법1 하락 +0.160(순위 5), 방법2 하락 -0.016(순위 9) → 부분 지지


### Grad-CAM: Transfer(E04, Full FT) vs Scratch(E01)

- 사진: 11번 노트북의 VLM 대상 중 정답이 2·7·9인 것. 클래스마다 `candidates.csv` 순서로 오분류 앞 2장과 정답 사례 1장(이미지를 보기 전에 정한 규칙)
- 각 모델이 **자기가 예측한 클래스**에 대해 본 곳입니다. 7×7 지도를 키운 것이라 거칩니다.
- 원본 사진이 들어가므로 공개 저장소에서는 그림 출력을 지웁니다.

In [7]:
import torch
from PIL import Image
from common import DATA_RAW, FIGURES_SAMPLES
from dataset import build_transforms
from predict import load_trained
from gradcam import gradcam

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cand = pd.read_csv(ROOT / "experiments" / "VLM" / "candidates.csv")
pick = pd.concat([pd.concat([cand[(cand.kind == "오분류") & (cand.target_id == t)].head(2), cand[(cand.kind == "정답") & (cand.target_id == t)].head(1)])
                  for t in (2, 7, 9)])
_, tf = build_transforms(224)
models = {"Transfer(E04)": load_trained("E04", "full", dev), "Scratch(E01)": load_trained("E01", "scratch", dev)}
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
fig, axes = plt.subplots(len(pick), 3, figsize=(11, 3.1 * len(pick)))
pad = []
for i, r in enumerate(pick.itertuples()):
    x = tf(Image.open(DATA_RAW / r.image_path).convert("RGB")).unsqueeze(0).to(dev)
    img = np.clip(x[0].permute(1, 2, 0).cpu().numpy() * STD + MEAN, 0, 1)
    axes[i, 0].imshow(img); axes[i, 0].set_title(f"정답 {r.true_name}({r.kind})", fontsize=9)
    for j, (nm, mdl) in enumerate(models.items(), start=1):
        cam, c = gradcam(mdl, x)
        pad.append({"모델": nm, "패딩 띠 CAM 비율": (cam[:49].sum() + cam[175:].sum()) / cam.sum()})
        axes[i, j].imshow(img); axes[i, j].imshow(cam, cmap="jet", alpha=0.45)
        axes[i, j].set_title(f"{nm} → {NAMES[c]}", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(FIGURES_SAMPLES / "gradcam_279_tl_vs_scratch.jpg", dpi=70); plt.show()

[원본 사진 그림 출력은 지웠습니다. 로컬 파일 figures/samples/에 저장되어 있고, 공개 저장소에는 넣지 않습니다(AI Hub 이용 조건).]


In [8]:
# 수치만(원본 사진 없음): 위아래 패딩 띠(49px씩)에 걸린 CAM 비율. 0에 가까울수록 패딩을 단서로 쓰지 않음
pd.DataFrame(pad).groupby("모델").mean().round(3)

,패딩 띠 CAM 비율
모델,
Scratch(E01),0.242
Transfer(E04),0.174


## 해석(2026-11-26)

### H1. 클립 평균: 규칙상 "지지". 8장 결합의 이득은 확인했으나 장수·다양성 효과는 미분리(독립 리뷰 보완)
- clip8은 세 fold 모두 올랐습니다(+0.060, +0.046, +0.037, 평균 +0.047). 공식 Val의 세 seed에서도 +0.050~+0.071입니다.
- 같은 카메라 사진만 평균한 cam4는 평균 +0.008에 그쳤습니다(fold 3은 0.000).
- **정정(2026-11-26)**: 처음에는 cam4를 "같은 카메라 4장"으로 보고, 이득이 다른 각도의 카메라를 합친 데서 나왔다고 해석했습니다. 그런데 클립은 4개 시점 × 시점마다 다른 카메라 2대(카메라 5대)라서, cam4는 실제로 1~2장 평균이었습니다. 장수 차이(8장 vs 1~2장)가 섞인 비교였습니다.
- 15번 노트북 A4에서 장수를 2장으로 맞추니, 다른 각도 2장(+0.009~+0.013)과 다른 순간 2장(+0.011~+0.017)의 이득이 구분되지 않았습니다. 그래서 **"한 각도만 보임"이 더 큰 한계라는 해석은 철회합니다.** 8장 결합의 개선이 관찰됐지만 장수 증가와 시점·시간 다양성의 개별 효과는 분리하지 못했습니다.
- 바벨 로우는 clip8로도 고르게 풀리지 않았습니다(fold 3: 0.27→0.12). 공식 Val(Z23)에서 크게 풀렸던 것(0.53→0.88)이 다른 사람에게는 일반화되지 않았습니다. 한 클립의 사진 대부분을 틀리면 평균해도 틀립니다.
- **기본 과제(사진 한 장)의 결과는 바뀌지 않습니다.** 클립 평균은 "여러 카메라·연속 프레임을 받을 수 있는 응용"에서 쓸 수 있는 방법으로 따로 보고합니다.

### H2. 사람을 크게(ccrop): CV1 "일관 개선형"이지만 붕괴 경향
- 세 fold 모두 올랐습니다(평균 +0.036). 대신 바벨 로우를 fold 2·3에서 잃었습니다(12번 노트북). 2단계 재검증 결과로 판정합니다.

### H3. 색 단서: 규칙상 "부분 지지". 내용상으로는 **색이 꼭 필요하지는 않다**는 쪽입니다
- 방법 1(컬러 모델에 흑백 입력)에서 덤벨 로우는 0.16 떨어졌지만 순위는 5위였습니다. 가장 크게 떨어진 것은 딥스(0.73)와 라잉 레그 레이즈(0.54)였습니다. 처음 보는 입력이라 생긴 분포 차이로, 색 단서와는 따로 봐야 합니다.
- 방법 2(처음부터 흑백으로 학습)에서는 덤벨 로우가 오히려 0.016 높았습니다. Macro F1도 컬러와 같았습니다(0.846 vs 0.845). **모델은 흑백에서도 같은 성능을 낼 수 있습니다.**
- 따라서 P1에서 흑백이 덤벨 로우를 크게 떨어뜨린 것(0.83→0.51)은 공식 Val의 한 사람(Z23)에 한정된 결과로 보입니다. VLM이 말한 "기구 식별의 어려움"은 색 문제가 아니라 **크기와 각도 문제**에 가깝습니다(H1·H2와 연결됨).

### Grad-CAM: Transfer는 사람을, Scratch는 사람 옆 바닥·배경을 자주 봅니다(정성)
- 9장 중 대부분에서 Transfer(E04)의 CAM은 사람의 몸통·다리에 모였습니다. Scratch(E01)는 여러 장에서 사람 오른쪽 아래의 바닥이나 배경 쪽에 모였습니다.
- 패딩 띠에 걸린 CAM 비율도 Scratch 0.24, Transfer 0.17입니다. 7×7 지도를 키운 것이라 경계가 번져서 정확한 값은 아닙니다. 그래도 Scratch가 사람 밖을 더 본다는 관찰과 같은 방향입니다.
- **배경 의존 가설**(EDA: 묶음마다 촬영 공간이 다름)과 맞습니다. Scratch가 Transfer보다 낮은 이유 중 하나일 수 있습니다. 9장 정성 관찰이므로 가설로만 둡니다.
- 두 모델 모두 손·덤벨처럼 작은 부위를 따로 짚지는 못했습니다. 7×7 해상도의 한계이기도 합니다.